# COM713 – Advanced Data Structures and Algorithms
## Part 2: Decentralised Medicine Supply Chain Optimisation System
### Sri Lanka Rural Hospital Network

**Module:** COM713 – Advanced Data Structures and Algorithms  
**Level:** Postgraduate  
**Focus:** Individual Implementation – Algorithm Design, Data Structures & Complexity Analysis  

---

## Problem Context (from Part 1)

Sri Lanka's current medical supply chain relies on a **centralised, top-down planning model** that lacks:
- Prediction of localised demand fluctuations
- Real-time inventory allocation across facilities
- Rapid, cost-effective emergency routing

This notebook implements a **Decentralised Inventory Optimisation and Emergency Routing System** using:

| Data Structure | Purpose | Complexity |
|---|---|---|
| Weighted Graph (dict of dicts) | Hospital network topology | O(V + E) space |
| HashMap (dict) | Inventory & distance lookups | O(1) average |
| Min-Heap (heapq) | Low-stock alerts **and** request-priority scheduling | O(log N) push/pop |
| Deque (collections.deque) | FIFO request arrival buffer & audit log | O(1) enqueue/dequeue |

**Primary Algorithm:** Dijkstra's Single-Source Shortest Path — O((V + E) log V)

**Beyond Part 1's original description**, this implementation makes two
refinements that change the system's actual behaviour, not just its
code structure — both directly grounded in Part 1's own algorithm
design (Sections 3.3 and 5.5):

- **Priority, not FIFO, request processing.** Pending requests are
  drained from the arrival deque into a Min-Heap keyed by urgency
  score, so the most critical shortage is served first regardless of
  arrival order (Section 6).
- **Safety-margin-aware supplier selection.** A candidate supplier must
  retain its own safety-stock level after donating — `available ≥
  quantity + supplier's own minimum stock` — so the system cannot
  "solve" one hospital's shortage by silently creating one at the
  supplier (Section 7.2).


In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# Section 1: Imports
# Standard library and third-party packages used throughout the notebook.
# ─────────────────────────────────────────────────────────────────────────────

from __future__ import annotations  # enable forward-ref type hints on Py 3.9

# Standard library
import csv                      # CSV file reading
import heapq                    # binary min-heap operations
import os                       # file path utilities
from collections import deque   # O(1) double-ended queue
from itertools import count     # tie-breaker generator for priority-heap entries
from typing import Dict, List, Optional, Tuple

# Visualisation
import matplotlib
matplotlib.use("Agg")  # non-interactive backend (required for headless/notebook execution)
import matplotlib.pyplot as plt          # plotting
import matplotlib.patches as mpatches   # legend handles
import networkx as nx                    # graph visualisation
import pandas as pd                      # data display

# Notebook display helpers
from IPython.display import display

print("✅ All imports successful")
print(f"   pandas   : {pd.__version__}")
print(f"   networkx : {nx.__version__}")


✅ All imports successful
   pandas   : 2.3.3
   networkx : 3.2.1


## Section 2 – Data Structures
### 2.1 `Hospital` Class (HashMap + Deque)

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# Hospital class
# ─────────────────────────────────────────────────────────────────────────────
# Data structures used INSIDE this class:
#   • dict  (HashMap)  – inventory and minimum-stock thresholds
#     → O(1) average time for get, set, contains operations
#   • collections.deque – circular buffer for transaction history
#     → O(1) append; automatically discards old entries (maxlen=50)
# ─────────────────────────────────────────────────────────────────────────────

class Hospital:
    """
    Represents a hospital node in the Sri Lanka medicine supply-chain network.

    Attributes
    ----------
    name         : str   – unique hospital identifier (graph key)
    inventory    : dict  – {medicine: quantity}  → HashMap O(1)
    min_stock    : dict  – {medicine: threshold} → HashMap O(1)
    history      : deque – last MAX_HISTORY transactions (circular buffer)
    """

    MAX_HISTORY = 50  # circular buffer limit

    def __init__(self, name: str, default_min_stock: int = 50):
        """
        Initialise a Hospital.

        Time  complexity : O(1)
        Space complexity : O(1) – empty dicts / deque at creation
        """
        self.name = name
        self.inventory: Dict[str, int] = {}   # HashMap
        self.min_stock: Dict[str, int] = {}   # HashMap for thresholds
        self._default_min = default_min_stock
        # Deque with maxlen acts as a circular buffer – O(1) append
        self.history: deque = deque(maxlen=self.MAX_HISTORY)

    # ── Inventory operations (all O(1) HashMap) ────────────────────────────

    def add_medicine(self, medicine: str, quantity: int,
                     min_stock: Optional[int] = None) -> None:
        """Add / overwrite a medicine entry. Time: O(1) | Space: O(1)."""
        self.inventory[medicine] = quantity
        self.min_stock[medicine] = (
            min_stock if min_stock is not None else self._default_min
        )
        self.history.append(f"[ADD] {medicine}: set to {quantity} units")

    def get_stock(self, medicine: str) -> int:
        """Return current stock. Time: O(1) – HashMap lookup."""
        return self.inventory.get(medicine, 0)

    def update_stock(self, medicine: str, quantity: int) -> None:
        """Update stock level and log the change. Time: O(1)."""
        old = self.inventory.get(medicine, 0)
        self.inventory[medicine] = quantity
        delta = quantity - old
        sign = "+" if delta >= 0 else ""
        self.history.append(f"[UPDATE] {medicine}: {old}→{quantity} ({sign}{delta})")

    # ── Alert / threshold helpers ──────────────────────────────────────────

    def get_min_stock(self, medicine: str) -> int:
        """Safety-stock threshold for a medicine. Time: O(1) – HashMap lookup."""
        return self.min_stock.get(medicine, self._default_min)

    def is_low_stock(self, medicine: str) -> bool:
        """True if stock ≤ minimum threshold. Time: O(1)."""
        return self.get_stock(medicine) <= self.get_min_stock(medicine)

    def get_shortage(self, medicine: str) -> int:
        """How many units short of the minimum threshold. Time: O(1)."""
        shortage = self.get_min_stock(medicine) - self.get_stock(medicine)
        return max(0, shortage)

    def urgency_score(self, medicine: str) -> float:
        """
        Urgency score = current_stock / min_threshold.
        Lower score = more critical. Used by BOTH the low-stock min-heap
        (Section 5) and the request-priority min-heap (Section 6).
        Time: O(1).
        """
        threshold = self.get_min_stock(medicine)
        if threshold == 0:
            return float('inf')
        return self.get_stock(medicine) / threshold

    def __str__(self) -> str:
        return f"Hospital({self.name})"

    def __repr__(self) -> str:
        return f"Hospital(name={self.name!r}, medicines={list(self.inventory.keys())})"


print("✅ Hospital class defined")
print("   Data structures: HashMap (inventory), HashMap (min_stock), Deque (history)")


✅ Hospital class defined
   Data structures: HashMap (inventory), HashMap (min_stock), Deque (history)


### 2.2 Dijkstra's Algorithm (Min-Heap + HashMap)

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# Dijkstra's Single-Source Shortest Path with path reconstruction
# ─────────────────────────────────────────────────────────────────────────────
#
# Data structures used:
#   • heapq (Binary Min-Heap) – priority queue for greedy node selection
#     → O(log V) push / pop
#   • dict distances       – O(1) distance lookup for each node
#   • dict predecessors    – O(1) predecessor lookup for path reconstruction
#
# Algorithm steps:
#   1. Initialise all distances to ∞; source = 0
#   2. Push (0, source) onto min-heap
#   3. Repeat until heap empty:
#        a. Pop minimum-distance node  ← O(log V)
#        b. Skip stale heap entries
#        c. Relax all outgoing edges; push improved distances ← O(log V)
#   4. Return distances + predecessors
#
# Time  complexity : O((V + E) log V)
# Space complexity : O(V + E)  – heap, distances, predecessors
# ─────────────────────────────────────────────────────────────────────────────

def dijkstra(graph: dict, start: str) -> Tuple[Dict, Dict]:
    """
    Run Dijkstra's algorithm from `start` on an adjacency dict graph.

    Parameters
    ----------
    graph : dict[str, dict[str, int]]
        Weighted undirected adjacency dictionary.
    start : str
        Source node key.

    Returns
    -------
    distances    : dict[str, float] – shortest distance to every node
    predecessors : dict[str, Optional[str]] – shortest-path tree
    """

    # ── Step 1: Initialise ────────────────────────────────────────────────
    # HashMap: node → shortest known distance (starts at ∞)
    distances: Dict[str, float] = {node: float('inf') for node in graph}
    distances[start] = 0

    # HashMap: node → predecessor on shortest path
    predecessors: Dict[str, Optional[str]] = {node: None for node in graph}

    # Min-Heap: (distance, node) – smallest distance always at index 0
    priority_queue: List[Tuple[float, str]] = [(0, start)]

    # ── Step 2-3: Main loop ───────────────────────────────────────────────
    while priority_queue:

        # Pop minimum-distance entry – O(log V)
        current_dist, current_node = heapq.heappop(priority_queue)

        # Stale entry guard: skip if we already found a shorter path
        if current_dist > distances[current_node]:
            continue

        # Relax all neighbours of current_node
        for neighbour, edge_weight in graph[current_node].items():
            new_dist = current_dist + edge_weight

            # Relaxation: update if shorter path found
            if new_dist < distances[neighbour]:
                distances[neighbour] = new_dist
                predecessors[neighbour] = current_node  # record predecessor
                heapq.heappush(priority_queue,          # O(log V)
                               (new_dist, neighbour))

    return distances, predecessors


def reconstruct_path(predecessors: dict, start: str,
                     destination: str) -> List[str]:
    """
    Reconstruct the shortest path by walking the predecessor map
    backwards from destination to start, then reversing.

    Time  complexity : O(V) – at most V steps
    Space complexity : O(V) – path list
    """
    path = []
    current = destination

    while current is not None:
        path.append(current)
        current = predecessors.get(current)
        if current == destination:  # cycle guard
            break

    path.reverse()

    # Return empty list if no valid path
    if path and path[0] == start:
        return path
    return []


print("✅ Dijkstra's algorithm defined")
print("   Time:  O((V + E) log V)")
print("   Space: O(V + E)")


✅ Dijkstra's algorithm defined
   Time:  O((V + E) log V)
   Space: O(V + E)


### 2.3 `SupplyChain` Class (Graph + Min-Heap + Deque)

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# SupplyChain – orchestrates the hospital network
# ─────────────────────────────────────────────────────────────────────────────
#
# Data structures used:
#   • dict  (HashMap)    – hospital registry and graph adjacency
#   • heapq (Min-Heap)   – low-stock alert queue sorted by urgency
#   • deque (FIFO Queue) – pending medicine request queue
#   • list               – transfer audit trail
# ─────────────────────────────────────────────────────────────────────────────

# Short name aliases (CSV uses full names; graph uses short keys)
# Note: "Dambulla" has no entry here and no inventory-CSV row. It appears
# only in hospital_routes.csv as a bare short name, so it loads purely as
# a graph routing waypoint — a regional distribution point (Part 1,
# Sec. 5.1: "V represents ... other relevant distribution points") rather
# than a hospital that holds its own medicine stock.
NAME_MAP = {
    "Kandy General Hospital":          "Kandy",
    "Matale District Hospital":        "Matale",
    "Kurunegala Teaching Hospital":    "Kurunegala",
    "Badulla General Hospital":        "Badulla",
    "Anuradhapura Teaching Hospital":  "Anuradhapura",
    "Polonnaruwa General Hospital":    "Polonnaruwa",
    "Nuwara Eliya District Hospital":  "Nuwara Eliya",
    "Ratnapura General Hospital":      "Ratnapura",
    "Monaragala District Hospital":    "Monaragala",
    "Hambantota General Hospital":     "Hambantota",
}


class SupplyChain:
    """
    Manages the decentralised hospital medicine supply-chain network.

    Internal state
    --------------
    hospitals      : dict  – HashMap: short_name → Hospital object
    graph          : dict  – Adjacency dict: node → {neighbour: weight}
    _alert_heap    : list  – Min-Heap of (urgency, hospital, medicine)
    _request_queue : deque – FIFO arrival buffer for pending requests;
                              drained into a priority Min-Heap at the
                              start of process_requests() (see below)
    _transfer_log  : list  – ordered audit trail
    """

    def __init__(self):
        self.hospitals: Dict[str, Hospital] = {}        # HashMap
        self.graph: Dict[str, Dict[str, int]] = {}      # Weighted adjacency dict
        self._alert_heap: list = []                     # Min-Heap
        self._request_queue: deque = deque()            # FIFO request queue
        self._transfer_log: list = []                   # Audit trail

    # ── Graph / hospital management ────────────────────────────────────────

    def add_hospital(self, hospital: Hospital) -> None:
        """Register a hospital node. Time: O(1) – HashMap insert."""
        self.hospitals[hospital.name] = hospital
        self.graph.setdefault(hospital.name, {})

    def add_route(self, node1: str, node2: str, distance: int) -> None:
        """Add undirected weighted edge. Time: O(1) – two HashMap inserts."""
        self.graph.setdefault(node1, {})[node2] = distance
        self.graph.setdefault(node2, {})[node1] = distance

    # ── CSV loaders ────────────────────────────────────────────────────────

    def load_inventory_csv(self, filepath: str) -> None:
        """
        Load hospital inventory from CSV.
        Time: O(H × M)  Space: O(H × M)
        """
        medicine_cols = ['Paracetamol', 'Amoxicillin', 'Insulin', 'Salbutamol', 'ORS']
        with open(filepath, newline='', encoding='utf-8') as f:
            for row in csv.DictReader(f):
                full_name = row['Hospital'].strip()
                if not full_name:
                    continue
                short_name = NAME_MAP.get(full_name, full_name)
                min_stock = int(row.get('Minimum Stock', 50) or 50)
                hospital = Hospital(short_name, default_min_stock=min_stock)
                for med in medicine_cols:
                    qty = int(row.get(med, 0) or 0)
                    hospital.add_medicine(med, qty, min_stock=min_stock)
                self.add_hospital(hospital)

    def load_routes_csv(self, filepath: str) -> None:
        """
        Load hospital routes from CSV.
        Time: O(E)  Space: O(V + E)
        """
        with open(filepath, newline='', encoding='utf-8') as f:
            for row in csv.DictReader(f):
                src = row['From'].strip()
                dst = row['To'].strip()
                dist_str = row['Distance (km)'].strip()
                if not src or not dst or not dist_str:
                    continue
                try:
                    self.add_route(src, dst, int(dist_str))
                except ValueError:
                    continue

    def load_requests_csv(self, filepath: str) -> None:
        """
        Load medicine requests into the FIFO deque.
        Time: O(R)  Space: O(R)
        """
        with open(filepath, newline='', encoding='utf-8') as f:
            for row in csv.DictReader(f):
                hospital_full = row['Hospital'].strip()
                if not hospital_full:
                    continue
                self._request_queue.append({   # deque.append = O(1)
                    'id':       row['Request ID'].strip(),
                    'hospital': NAME_MAP.get(hospital_full, hospital_full),
                    'medicine': row['Medicine'].strip(),
                    'quantity': int(row['Quantity'].strip()),
                })

    # ── Core algorithm: Dijkstra-based supplier search ─────────────────────

    def find_supplier(self, requesting_hospital: str,
                      medicine: str, quantity: int) -> tuple:
        """
        Find the nearest hospital able to supply `quantity` units
        of `medicine` using Dijkstra's SSSP.

        Algorithm
        ---------
        1. Run Dijkstra from requesting_hospital  → O((V+E) log V)
        2. Filter hospitals that can supply WITHOUT breaching their own
           safety stock: available_stock ≥ quantity + supplier_min_stock
           (Part 1, Sec. 5.5 – Surplus-Facility Selection). This stops
           the system curing one hospital's shortage by creating a new
           one at the supplier.                     → O(H)
        3. Select minimum-distance feasible hospital → O(H)
        4. Reconstruct path                          → O(V)

        Overall time : O((V + E) log V)
        """
        if requesting_hospital not in self.graph:
            return None, float('inf'), []

        # Dijkstra's – returns distances and predecessor map
        distances, predecessors = dijkstra(self.graph, requesting_hospital)

        best_hospital = None
        best_distance = float('inf')

        # Greedy selection: nearest hospital that can supply the request
        # while staying at or above its own safety-stock threshold
        for h_name, hospital in self.hospitals.items():
            if h_name == requesting_hospital:
                continue
            safety_stock = hospital.get_min_stock(medicine)          # O(1)
            if hospital.get_stock(medicine) >= quantity + safety_stock:  # O(1)
                dist = distances.get(h_name, float('inf'))    # O(1)
                if dist < best_distance:
                    best_distance = dist
                    best_hospital = hospital

        if best_hospital is None:
            return None, float('inf'), []

        # Reconstruct the actual route
        path = reconstruct_path(predecessors, requesting_hospital,
                                best_hospital.name)
        return best_hospital, best_distance, path

    # ── Transfer execution ─────────────────────────────────────────────────

    def transfer(self, source: Hospital, destination: Hospital,
                 medicine: str, quantity: int) -> dict:
        """
        Execute a medicine transfer and log the event.
        Time: O(1) – two HashMap updates
        """
        src_before = source.get_stock(medicine)
        dst_before = destination.get_stock(medicine)

        source.update_stock(medicine, src_before - quantity)         # O(1)
        destination.update_stock(medicine, dst_before + quantity)    # O(1)

        record = {
            'source': source.name, 'destination': destination.name,
            'medicine': medicine,  'quantity': quantity,
            'src_before': src_before, 'src_after': src_before - quantity,
            'dst_before': dst_before, 'dst_after': dst_before + quantity,
        }
        self._transfer_log.append(record)
        return record

    # ── Min-Heap alert system ──────────────────────────────────────────────

    def get_low_stock_alerts(self) -> List[dict]:
        """
        Build a min-heap of all low-stock situations and return them
        ordered by urgency (most critical first).

        Min-Heap property: parent ≤ children (by urgency_score)
        → heappush: O(log N)  |  heappop: O(log N)
        Building heap for N items: O(N log N)
        """
        heap = []
        # Build the heap by pushing all low-stock items
        for h_name, hospital in self.hospitals.items():
            for medicine in hospital.inventory:
                score = hospital.urgency_score(medicine)   # O(1)
                if score <= 1.0:                           # at or below threshold
                    heapq.heappush(heap, (score, h_name, medicine))  # O(log N)

        # Pop all in priority order (most critical = lowest score first)
        alerts = []
        while heap:
            score, h_name, medicine = heapq.heappop(heap)   # O(log N)
            hospital = self.hospitals[h_name]
            alerts.append({
                'hospital': h_name, 'medicine': medicine,
                'stock': hospital.get_stock(medicine),
                'min_stock': hospital.min_stock.get(medicine, 0),
                'shortage': hospital.get_shortage(medicine),
                'urgency_score': round(score, 3),
            })
        return alerts

    # ── Priority-queue request processor ───────────────────────────────────

    def process_requests(self) -> List[dict]:
        """
        Process all pending requests in PRIORITY order, not arrival
        (FIFO) order.

        Rationale (Part 1, Sec. 3.3 / 5.3): a strict first-come,
        first-served queue can serve a mildly low hospital before a
        critically empty one that simply arrived later. Instead, every
        pending request is scored with the requesting hospital's
        urgency_score() for that medicine (lower = more critical) and
        served from a Min-Heap, so the most urgent shortage is always
        handled first regardless of arrival order.

        Algorithm
        ---------
        1. Drain the FIFO arrival deque                    – O(R)
        2. Push (urgency, arrival_seq, request) onto a
           Min-Heap. arrival_seq is a tie-breaker that keeps
           equally-urgent requests in arrival order and stops
           heapq from ever comparing the request dicts.      – O(R log R)
        3. Pop requests strictly in priority order           – O(log R) each
        4. Find nearest FEASIBLE supplier via Dijkstra        – O((V+E) log V)
        5. Execute the transfer if a supplier was found       – O(1)

        Total time : O(R log R + R·(V+E) log V) — the heap overhead is
        negligible next to the R Dijkstra searches, so this remains
        dominated by O(R·(V+E) log V), same order as plain FIFO.
        """
        results = []

        # Step 1-2: drain FIFO arrivals into a priority Min-Heap
        priority_heap = []
        sequence = count()  # tie-breaker; also preserves arrival order
        while self._request_queue:
            req = self._request_queue.popleft()   # FIFO dequeue O(1)
            h_name, medicine = req['hospital'], req['medicine']
            if h_name in self.hospitals:
                urgency = self.hospitals[h_name].urgency_score(medicine)  # O(1)
            else:
                # Unknown hospital: surface the data error immediately
                # instead of letting it hide behind real shortages.
                urgency = float('-inf')
            heapq.heappush(priority_heap, (urgency, next(sequence), req))  # O(log R)

        # Step 3: process strictly in priority order (most critical first)
        while priority_heap:
            _, _, req = heapq.heappop(priority_heap)   # O(log R)
            r_id, h_name, medicine, quantity = (
                req['id'], req['hospital'], req['medicine'], req['quantity']
            )

            if h_name not in self.hospitals:
                results.append({'request_id': r_id, 'status': 'FAILED – hospital not found',
                                'hospital': h_name, 'medicine': medicine, 'quantity': quantity})
                continue

            destination = self.hospitals[h_name]
            supplier, distance, path = self.find_supplier(h_name, medicine, quantity)

            if supplier:
                record = self.transfer(supplier, destination, medicine, quantity)
                results.append({
                    'request_id': r_id, 'status': 'SUCCESS',
                    'hospital': h_name, 'medicine': medicine, 'quantity': quantity,
                    'supplier': supplier.name, 'distance_km': distance,
                    'path': ' → '.join(path) if path else 'direct',
                    **record,
                })
            else:
                results.append({
                    'request_id': r_id, 'status': 'FAILED – no supplier',
                    'hospital': h_name, 'medicine': medicine, 'quantity': quantity,
                    'supplier': None, 'distance_km': None, 'path': None,
                })
        return results


print("✅ SupplyChain class defined")
print("   Data structures: HashMap (hospitals), Graph (adjacency dict),")
print("                    Min-Heap (alerts + request priority),")
print("                    Deque (FIFO request arrival buffer), List (audit)")


✅ SupplyChain class defined
   Data structures: HashMap (hospitals), Graph (adjacency dict),
                    Min-Heap (alerts + request priority),
                    Deque (FIFO request arrival buffer), List (audit)


## Section 3 – Dataset Loading
### 3.1 Load CSV Files

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# Load real CSV datasets
# ─────────────────────────────────────────────────────────────────────────────
# Datasets:
#   hospital_inventory.csv  – stock levels per hospital per medicine
#   hospital_routes.csv     – distances (km) between hospitals
#   medicine_requests.csv   – pending supply requests
#
# Source: Simulated dataset based on Sri Lanka Ministry of Health
# facility network. Reference: Ministry of Health Sri Lanka (2024),
# "National Medicine Regulatory Authority Reports".
# Dataset URL: https://www.health.gov.lk (publicly available reports)
# ─────────────────────────────────────────────────────────────────────────────

# Resolve paths relative to this notebook's directory
BASE_DIR = os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else os.getcwd()

INVENTORY_CSV = os.path.join(BASE_DIR, "hospital_inventory.csv")
ROUTES_CSV    = os.path.join(BASE_DIR, "hospital_routes.csv")
REQUESTS_CSV  = os.path.join(BASE_DIR, "medicine_requests.csv")

# Initialise and load the supply chain
system = SupplyChain()
system.load_inventory_csv(INVENTORY_CSV)
system.load_routes_csv(ROUTES_CSV)
system.load_requests_csv(REQUESTS_CSV)

print(f"✅ Loaded {len(system.hospitals)} hospitals")
print(f"✅ Loaded {sum(len(v) for v in system.graph.values()) // 2} routes")
print(f"✅ Loaded {len(system._request_queue)} pending requests")


✅ Loaded 10 hospitals
✅ Loaded 12 routes
✅ Loaded 5 pending requests


### 3.2 Hospital Inventory (Initial State)

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# Display the initial hospital inventory as a formatted pandas DataFrame.
# ─────────────────────────────────────────────────────────────────────────────

# Build inventory DataFrame from the HashMap-based hospital objects
inventory_data = []
for h_name, hospital in sorted(system.hospitals.items()):
    row = {'Hospital': h_name, 'Min Stock': hospital._default_min}
    row.update(hospital.inventory)  # add each medicine column from HashMap
    inventory_data.append(row)

df_inventory = pd.DataFrame(inventory_data).set_index('Hospital')
df_inventory = df_inventory.fillna(0).astype(int)

# Highlight low-stock values
def highlight_low(val, threshold=50):
    """Apply red background to values at or below threshold."""
    return 'background-color: #ffcccc; font-weight: bold' if val <= threshold else ''

print("Initial Hospital Inventory:")
display(df_inventory.style
        .map(highlight_low, subset=['Insulin'])
        .set_caption("⚠ Red = at or below minimum stock threshold")
        .format("{:,}"))


Initial Hospital Inventory:


,Min Stock,Paracetamol,Amoxicillin,Insulin,Salbutamol,ORS
Hospital,,,,,,
Anuradhapura,100,520,260,90,150,320
Badulla,60,280,130,25,80,170
Hambantota,80,380,190,45,100,250
Kandy,100,500,250,80,120,300
Kurunegala,100,450,220,60,90,280
Matale,50,300,100,5,70,150
Monaragala,50,210,80,8,55,120
Nuwara Eliya,50,240,95,12,65,140
Polonnaruwa,60,260,110,15,75,180


### 3.3 Hospital Routes

In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# Display all hospital routes as a pandas DataFrame.
# The graph is represented as an adjacency dict (dict of dicts).
# ─────────────────────────────────────────────────────────────────────────────

routes_data = []
seen = set()   # track undirected edges to avoid duplicates
for src, neighbours in system.graph.items():
    for dst, dist in neighbours.items():
        edge_key = tuple(sorted([src, dst]))
        if edge_key not in seen:
            seen.add(edge_key)
            routes_data.append({
                'From': src,
                'To': dst,
                'Distance (km)': dist
            })

df_routes = pd.DataFrame(routes_data).sort_values('Distance (km)').reset_index(drop=True)
print(f"Hospital Route Network ({len(df_routes)} routes):")
display(df_routes.style.background_gradient(subset=['Distance (km)'], cmap='YlOrRd'))


Hospital Route Network (12 routes):


,From,To,Distance (km)
0,Kandy,Matale,25
1,Kandy,Kurunegala,42
2,Matale,Dambulla,45
3,Badulla,Monaragala,58
4,Anuradhapura,Dambulla,65
5,Kandy,Nuwara Eliya,75
6,Monaragala,Hambantota,82
7,Kurunegala,Anuradhapura,88
8,Kandy,Ratnapura,92
9,Anuradhapura,Polonnaruwa,105


### 3.4 Pending Medicine Requests

In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# Display the pending medicine requests: arrival order (FIFO) AND a preview
# of the priority order they will actually be processed in (Section 6).
# ─────────────────────────────────────────────────────────────────────────────

# Peek into the deque (non-destructive) by converting to list
requests_data = [
    {'Request ID': r['id'], 'Hospital': r['hospital'],
     'Medicine': r['medicine'], 'Quantity': r['quantity']}
    for r in system._request_queue
]
df_requests = pd.DataFrame(requests_data)
print("Pending Medicine Requests — arrival (FIFO) order:")
display(df_requests)

# Preview the priority order WITHOUT consuming the deque, using the same
# urgency_score() the Min-Heap in process_requests() will use — lower
# score = more critical, so it is served first regardless of arrival order.
preview_rows = []
for r in system._request_queue:
    hosp = system.hospitals.get(r['hospital'])
    urgency = hosp.urgency_score(r['medicine']) if hosp else float('-inf')
    preview_rows.append({
        'Request ID': r['id'], 'Hospital': r['hospital'],
        'Medicine': r['medicine'], 'Quantity': r['quantity'],
        'Urgency Score': round(urgency, 3),
    })
df_priority_preview = (pd.DataFrame(preview_rows)
                        .sort_values('Urgency Score')
                        .reset_index(drop=True))
df_priority_preview.index = range(1, len(df_priority_preview) + 1)
df_priority_preview.index.name = 'Priority Rank'

print("\nSame requests — PRIORITY (Min-Heap) order they will be served in:")
print("(lower Urgency Score = closer to/below the hospital's own safety-stock threshold = more critical)")
display(df_priority_preview.style
        .background_gradient(subset=['Urgency Score'], cmap='RdYlGn'))


Pending Medicine Requests — arrival (FIFO) order:


,Request ID,Hospital,Medicine,Quantity
0,1,Matale,Insulin,20
1,2,Monaragala,Insulin,15
2,3,Nuwara Eliya,Amoxicillin,50
3,4,Polonnaruwa,Salbutamol,30
4,5,Badulla,ORS,80



Same requests — PRIORITY (Min-Heap) order they will be served in:
(lower Urgency Score = closer to/below the hospital's own safety-stock threshold = more critical)


,Request ID,Hospital,Medicine,Quantity,Urgency Score
Priority Rank,,,,,
1,1,Matale,Insulin,20,0.100000
2,2,Monaragala,Insulin,15,0.160000
3,4,Polonnaruwa,Salbutamol,30,1.250000
4,3,Nuwara Eliya,Amoxicillin,50,1.900000
5,5,Badulla,ORS,80,2.833000


## Section 4 – Network Visualisation
### 4.1 Hospital Route Graph

In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# Visualise the hospital network as a weighted graph using NetworkX.
# Node colour indicates Insulin stock level (red = critical, green = good).
# ─────────────────────────────────────────────────────────────────────────────

# Build a NetworkX graph from the adjacency dict
G = nx.Graph()
for src, neighbours in system.graph.items():
    for dst, dist in neighbours.items():
        G.add_edge(src, dst, weight=dist)

# Colour nodes by Insulin urgency score
node_colors = []
for node in G.nodes():
    if node in system.hospitals:
        score = system.hospitals[node].urgency_score('Insulin')
        # Red (critical) → Green (adequate)
        if score < 0.3:
            node_colors.append('#e74c3c')    # critical red
        elif score < 0.7:
            node_colors.append('#e67e22')    # warning orange
        elif score <= 1.0:
            node_colors.append('#f1c40f')    # caution yellow
        else:
            node_colors.append('#27ae60')    # adequate green
    else:
        node_colors.append('#95a5a6')        # grey (waypoint only)

# Layout
pos = nx.spring_layout(G, seed=42, k=2.2)

fig, ax = plt.subplots(figsize=(14, 9))
ax.set_facecolor('#1a1a2e')
fig.patch.set_facecolor('#1a1a2e')

# Draw edges
nx.draw_networkx_edges(G, pos, ax=ax, edge_color='#4a4a8a',
                       width=2, alpha=0.7)
# Edge labels (distances)
edge_labels = nx.get_edge_attributes(G, 'weight')
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels,
                             font_size=8, font_color='#a0a0c0', ax=ax)
# Nodes
nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors,
                       node_size=1200, alpha=0.95)
# Node labels
nx.draw_networkx_labels(G, pos, ax=ax, font_size=8,
                        font_color='white', font_weight='bold')

# Legend
legend_items = [
    mpatches.Patch(color='#e74c3c', label='Critical (< 30% of min)'),
    mpatches.Patch(color='#e67e22', label='Warning (30–70% of min)'),
    mpatches.Patch(color='#f1c40f', label='Caution (70–100% of min)'),
    mpatches.Patch(color='#27ae60', label='Adequate (> 100% of min)'),
]
ax.legend(handles=legend_items, loc='lower left',
          facecolor='#2d2d5e', labelcolor='white', fontsize=9)

ax.set_title("Sri Lanka Hospital Network – Insulin Stock Status\n(Node colour = urgency level | Edge labels = distance in km)",
             color='white', fontsize=13, pad=15)
ax.axis('off')
plt.tight_layout()
plt.savefig("network_graph.png", dpi=150, bbox_inches='tight',
            facecolor='#1a1a2e')
plt.show()
print("✅ Network graph saved as network_graph.png")


✅ Network graph saved as network_graph.png


/var/folders/c4/p5_t0y6x7jg16h0gg8x7kwx80000gn/T/ipykernel_12945/1624366772.py:66: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section 5 – Low-Stock Alerts (Min-Heap)
### 5.1 Priority Queue Alert System

In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# Low-Stock Alert System using a Min-Heap
# ─────────────────────────────────────────────────────────────────────────────
#
# The min-heap ensures the MOST CRITICAL hospital/medicine combination
# is always at the top (index 0) of the heap.
#
# Urgency Score = current_stock / minimum_threshold
#   → Lower score = more urgent (less stock relative to what is needed)
#   → Score = 0.1 means only 10% of minimum stock remains
#
# heapq in Python implements a MIN-heap:
#   • heappush(heap, item) : O(log N) – maintains heap property
#   • heappop(heap)        : O(log N) – returns minimum item
#   • Building N-item heap : O(N log N)
# ─────────────────────────────────────────────────────────────────────────────

alerts = system.get_low_stock_alerts()

df_alerts = pd.DataFrame(alerts)
df_alerts.index = range(1, len(df_alerts) + 1)
df_alerts.index.name = 'Priority'
df_alerts.columns = ['Hospital', 'Medicine', 'Stock', 'Min Stock', 'Shortage', 'Urgency Score']

print(f"Low-Stock Alerts ({len(df_alerts)} items) — most critical first:")
display(df_alerts.style
        .background_gradient(subset=['Urgency Score'], cmap='RdYlGn')
        .bar(subset=['Shortage'], color='#e74c3c', width=60)
        .format({'Urgency Score': '{:.3f}', 'Stock': '{:,}',
                 'Min Stock': '{:,}', 'Shortage': '{:,}'}))


Low-Stock Alerts (11 items) — most critical first:


,Hospital,Medicine,Stock,Min Stock,Shortage,Urgency Score
Priority,,,,,,
1,Matale,Insulin,5,50,45,0.100
2,Monaragala,Insulin,8,50,42,0.160
3,Nuwara Eliya,Insulin,12,50,38,0.240
4,Polonnaruwa,Insulin,15,60,45,0.250
5,Badulla,Insulin,25,60,35,0.417
6,Ratnapura,Insulin,35,70,35,0.500
7,Hambantota,Insulin,45,80,35,0.562
8,Kurunegala,Insulin,60,100,40,0.600
9,Kandy,Insulin,80,100,20,0.800


## Section 6 – Processing Medicine Requests (Priority Order, not FIFO)
### 6.1 Priority Min-Heap (drained from the FIFO arrival deque) + Dijkstra's Algorithm

In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# Process all medicine requests in PRIORITY order (Min-Heap), not FIFO.
# For each request, Dijkstra's algorithm finds the nearest supplier that
# can fulfil it WITHOUT breaching the supplier's own safety-stock level.
# ─────────────────────────────────────────────────────────────────────────────
#
# Request processing (see process_requests() in Section 2.3):
#   • FIFO deque drained into a Min-Heap keyed by urgency_score() – O(R log R)
#   • heappop() extracts the most critical request first             – O(log R)
#   • For each request: Dijkstra O((V+E) log V) + transfer O(1)
#   • Total: O(R log R + R × (V+E) log V), dominated by the Dijkstra term
# ─────────────────────────────────────────────────────────────────────────────

print("Processing medicine requests in priority order (most critical first)...\n")

results = system.process_requests()

for r in results:
    status_icon = "✅" if r['status'] == 'SUCCESS' else "❌"
    print(f"{status_icon} Request #{r['request_id']}")
    print(f"   Hospital   : {r['hospital']}")
    print(f"   Medicine   : {r['medicine']}  |  Quantity: {r['quantity']} units")
    print(f"   Status     : {r['status']}")
    if r.get('supplier'):
        print(f"   Supplier   : {r['supplier']}  ({r['distance_km']} km away)")
        print(f"   Route      : {r['path']}")
        print(f"   Stock      : {r['supplier']} {r['medicine']}: {r['src_before']} → {r['src_after']} units")
        print(f"              : {r['hospital']} {r['medicine']}: {r['dst_before']} → {r['dst_after']} units")
    print()


Processing medicine requests in priority order (most critical first)...

❌ Request #1
   Hospital   : Matale
   Medicine   : Insulin  |  Quantity: 20 units
   Status     : FAILED – no supplier

❌ Request #2
   Hospital   : Monaragala
   Medicine   : Insulin  |  Quantity: 15 units
   Status     : FAILED – no supplier

✅ Request #4
   Hospital   : Polonnaruwa
   Medicine   : Salbutamol  |  Quantity: 30 units
   Status     : SUCCESS
   Supplier   : Anuradhapura  (105 km away)
   Route      : Polonnaruwa → Anuradhapura
   Stock      : Anuradhapura Salbutamol: 150 → 120 units
              : Polonnaruwa Salbutamol: 75 → 105 units

✅ Request #3
   Hospital   : Nuwara Eliya
   Medicine   : Amoxicillin  |  Quantity: 50 units
   Status     : SUCCESS
   Supplier   : Kandy  (75 km away)
   Route      : Nuwara Eliya → Kandy
   Stock      : Kandy Amoxicillin: 250 → 200 units
              : Nuwara Eliya Amoxicillin: 95 → 145 units

✅ Request #5
   Hospital   : Badulla
   Medicine   : ORS  |  Quanti

### 6.2 Results Summary Table

In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# Display the transfer results as a formatted pandas table.
# ─────────────────────────────────────────────────────────────────────────────

df_results = pd.DataFrame([{
    'Served #':  i,
    'Req ID':    r['request_id'],
    'Hospital':  r['hospital'],
    'Medicine':  r['medicine'],
    'Quantity':  r['quantity'],
    'Status':    r['status'],
    'Supplier':  r.get('supplier', 'N/A'),
    'Dist (km)': r.get('distance_km', 'N/A'),
    'Route':     r.get('path', 'N/A'),
} for i, r in enumerate(results, start=1)]).set_index('Served #')

n_success = sum(1 for r in results if r['status'] == 'SUCCESS')
display(df_results.style
        .map(lambda v: 'color: #27ae60; font-weight:bold'
                  if v == 'SUCCESS' else
                  'color: #e74c3c; font-weight:bold', subset=['Status'])
        .set_caption(f"Medicine Transfer Results – {len(results)} Requests, "
                     f"processed in priority order ({n_success} succeeded)"))


,Req ID,Hospital,Medicine,Quantity,Status,Supplier,Dist (km),Route
Served #,,,,,,,,
1,1,Matale,Insulin,20,FAILED – no supplier,None,nan,None
2,2,Monaragala,Insulin,15,FAILED – no supplier,None,nan,None
3,4,Polonnaruwa,Salbutamol,30,SUCCESS,Anuradhapura,105.000000,Polonnaruwa → Anuradhapura
4,3,Nuwara Eliya,Amoxicillin,50,SUCCESS,Kandy,75.000000,Nuwara Eliya → Kandy
5,5,Badulla,ORS,80,SUCCESS,Polonnaruwa,115.000000,Badulla → Polonnaruwa


### 6.3 Inventory Before vs After

In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# Visualise how inventory levels changed after all transfers.
# Uses the transfer audit log stored in system._transfer_log.
#
# Robust to ANY number of successful transfers (0, 1, or many): with the
# priority + safety-margin logic some requests can legitimately fail
# (e.g. a nationwide shortage where no hospital can donate without
# breaching its own safety stock — see Section 7), so this must not
# assume a fixed request count.
# ─────────────────────────────────────────────────────────────────────────────

n_transfers = len(system._transfer_log)

if n_transfers == 0:
    fig, ax = plt.subplots(figsize=(8, 4))
    fig.patch.set_facecolor('#1a1a2e')
    ax.set_facecolor('#16213e')
    ax.text(0.5, 0.5, 'No successful transfers to display',
            ha='center', va='center', color='white', fontsize=12)
    ax.axis('off')
else:
    fig, axes = plt.subplots(1, n_transfers, figsize=(16, 5))
    fig.patch.set_facecolor('#1a1a2e')
    if n_transfers == 1:
        axes = [axes]  # plt.subplots returns a bare Axes (not a list) when ncols=1

    for ax, record in zip(axes, system._transfer_log):
        hospitals = [record['source'], record['destination']]
        before    = [record['src_before'], record['dst_before']]
        after     = [record['src_after'],  record['dst_after']]

        x = range(len(hospitals))
        bars_before = ax.bar([i - 0.2 for i in x], before, 0.35,
                             label='Before', color='#e74c3c', alpha=0.85)
        bars_after  = ax.bar([i + 0.2 for i in x], after, 0.35,
                             label='After',  color='#27ae60', alpha=0.85)

        # Value labels
        for bar in bars_before:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                    str(int(bar.get_height())), ha='center', va='bottom',
                    fontsize=8, color='white')
        for bar in bars_after:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                    str(int(bar.get_height())), ha='center', va='bottom',
                    fontsize=8, color='white')

        ax.set_facecolor('#16213e')
        ax.set_xticks(list(x))
        ax.set_xticklabels(hospitals, fontsize=8, color='white')
        ax.tick_params(colors='white')
        ax.set_title(f"{record['medicine']}\n({record['quantity']} units)",
                     fontsize=9, color='white')
        ax.spines['bottom'].set_color('#4a4a8a')
        ax.spines['left'].set_color('#4a4a8a')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    axes[0].legend(facecolor='#2d2d5e', labelcolor='white', fontsize=8)

fig.suptitle(f"Inventory Changes After {n_transfers} Successful Transfer(s) "
             "(Red = Before | Green = After)",
             color='white', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig("inventory_changes.png", dpi=150, bbox_inches='tight',
            facecolor='#1a1a2e')
plt.show()
print("✅ Inventory chart saved as inventory_changes.png")


✅ Inventory chart saved as inventory_changes.png


/var/folders/c4/p5_t0y6x7jg16h0gg8x7kwx80000gn/T/ipykernel_12945/1466935218.py:67: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section 7 – Algorithm Deep-Dive
### 7.1 Dijkstra's Algorithm – Step-by-Step Trace

In [14]:
# ─────────────────────────────────────────────────────────────────────────────
# Dijkstra's Algorithm – Annotated Step-by-Step Trace
# Runs Dijkstra from Matale and shows how distances are discovered.
# ─────────────────────────────────────────────────────────────────────────────

def dijkstra_traced(graph: dict, start: str):
    """
    Identical to dijkstra() but prints each relaxation step.
    Used to demonstrate the algorithm's operation clearly.
    """
    distances = {node: float('inf') for node in graph}
    predecessors = {node: None for node in graph}
    distances[start] = 0

    priority_queue = [(0, start)]

    print(f"  Starting from: {start}")
    print(f"  Initial distances: all = ∞ except {start} = 0\n")
    step = 0

    while priority_queue:
        current_dist, current_node = heapq.heappop(priority_queue)

        if current_dist > distances[current_node]:
            print(f"  [Skip] {current_node} (stale entry, dist={current_dist})")
            continue

        step += 1
        print(f"  Step {step}: Visiting '{current_node}' (dist={current_dist} km)")

        for neighbour, weight in graph[current_node].items():
            new_dist = current_dist + weight
            if new_dist < distances[neighbour]:
                old = distances[neighbour]
                distances[neighbour] = new_dist
                predecessors[neighbour] = current_node
                heapq.heappush(priority_queue, (new_dist, neighbour))
                inf_str = "∞" if old == float('inf') else str(old)
                print(f"    ↳ Relax edge {current_node}→{neighbour}: "
                      f"dist {inf_str} → {new_dist} km  ✓ updated")
            else:
                print(f"    ↳ Edge {current_node}→{neighbour}: "
                      f"new_dist={new_dist} ≥ current={distances[neighbour]}  (no update)")

    print("\n  Final shortest distances from", start)
    for node, dist in sorted(distances.items()):
        d = str(dist) + " km" if dist != float('inf') else "∞ (unreachable)"
        path = reconstruct_path(predecessors, start, node)
        route = " → ".join(path) if path else "N/A"
        print(f"    {node:<20}: {d:<15}  path: {route}")

    return distances, predecessors


print("=" * 60)
print("  Dijkstra Trace: Shortest paths from Matale")
print("=" * 60)
dijkstra_traced(system.graph, "Matale")


  Dijkstra Trace: Shortest paths from Matale
  Starting from: Matale
  Initial distances: all = ∞ except Matale = 0

  Step 1: Visiting 'Matale' (dist=0 km)
    ↳ Relax edge Matale→Kandy: dist ∞ → 25 km  ✓ updated
    ↳ Relax edge Matale→Dambulla: dist ∞ → 45 km  ✓ updated
  Step 2: Visiting 'Kandy' (dist=25 km)
    ↳ Edge Kandy→Matale: new_dist=50 ≥ current=0  (no update)
    ↳ Relax edge Kandy→Nuwara Eliya: dist ∞ → 100 km  ✓ updated
    ↳ Relax edge Kandy→Kurunegala: dist ∞ → 67 km  ✓ updated
    ↳ Relax edge Kandy→Ratnapura: dist ∞ → 117 km  ✓ updated
  Step 3: Visiting 'Dambulla' (dist=45 km)
    ↳ Edge Dambulla→Matale: new_dist=90 ≥ current=0  (no update)
    ↳ Relax edge Dambulla→Anuradhapura: dist ∞ → 110 km  ✓ updated
  Step 4: Visiting 'Kurunegala' (dist=67 km)
    ↳ Edge Kurunegala→Kandy: new_dist=109 ≥ current=25  (no update)
    ↳ Edge Kurunegala→Anuradhapura: new_dist=155 ≥ current=110  (no update)
  Step 5: Visiting 'Nuwara Eliya' (dist=100 km)
    ↳ Edge Nuwara Eliya→Ka

({'Kandy': 25,
  'Matale': 0,
  'Kurunegala': 67,
  'Badulla': 330,
  'Anuradhapura': 110,
  'Polonnaruwa': 215,
  'Nuwara Eliya': 100,
  'Ratnapura': 117,
  'Monaragala': 339,
  'Hambantota': 257,
  'Dambulla': 45},
 {'Kandy': 'Matale',
  'Matale': None,
  'Kurunegala': 'Kandy',
  'Badulla': 'Polonnaruwa',
  'Anuradhapura': 'Dambulla',
  'Polonnaruwa': 'Anuradhapura',
  'Nuwara Eliya': 'Kandy',
  'Ratnapura': 'Kandy',
  'Monaragala': 'Hambantota',
  'Hambantota': 'Ratnapura',
  'Dambulla': 'Matale'})

### 7.2 Why Safety-Margin-Aware Supplier Selection Matters

Section 6 showed both Insulin requests (Matale, Monaragala) **failing** —
`FAILED – no supplier`. This is not a bug: it is the safety-margin check
in `find_supplier()` (Part 1, Sec. 5.5) correctly refusing to rob one
hospital's emergency stock to patch another, because Insulin is short
**everywhere** in this dataset — every hospital's own Insulin stock is
already below its own minimum-stock threshold.

The cell below replays both Insulin requests through the **old naive
rule** (`stock ≥ quantity`, no safety margin — i.e. exactly what
`find_supplier()` did before this refinement) and compares it against
the current safety-margin-aware result, to make the effect of the fix
concrete rather than just asserted.

In [15]:
# ─────────────────────────────────────────────────────────────────────────────
# Before / after comparison: naive "stock ≥ quantity" supplier selection
# vs. the safety-margin-aware selection now used in find_supplier().
#
# Both insulin requests are replayed here purely for comparison — this
# does NOT execute any transfer, so it does not mutate system state.
# ─────────────────────────────────────────────────────────────────────────────

def find_supplier_naive(chain, requesting_hospital, medicine, quantity):
    """
    Reproduces the ORIGINAL find_supplier() logic: any hospital with
    stock >= quantity is eligible, regardless of its own safety stock.
    Time complexity: O((V+E) log V), identical shape to the current
    safety-margin-aware version — only the feasibility test differs.
    """
    if requesting_hospital not in chain.graph:
        return None, float('inf'), []
    distances, predecessors = dijkstra(chain.graph, requesting_hospital)
    best_hospital, best_distance = None, float('inf')
    for h_name, hospital in chain.hospitals.items():
        if h_name == requesting_hospital:
            continue
        if hospital.get_stock(medicine) >= quantity:          # <- no safety margin
            dist = distances.get(h_name, float('inf'))
            if dist < best_distance:
                best_distance, best_hospital = dist, hospital
    if best_hospital is None:
        return None, float('inf'), []
    path = reconstruct_path(predecessors, requesting_hospital, best_hospital.name)
    return best_hospital, best_distance, path


insulin_requests = [('Matale', 20), ('Monaragala', 15)]
comparison_rows = []

for hosp_name, qty in insulin_requests:
    naive_supplier, naive_dist, _ = find_supplier_naive(system, hosp_name, 'Insulin', qty)
    safe_supplier, safe_dist, _   = system.find_supplier(hosp_name, 'Insulin', qty)

    if naive_supplier is not None:
        stock_before = naive_supplier.get_stock('Insulin')
        threshold    = naive_supplier.get_min_stock('Insulin')
        stock_after  = stock_before - qty
        naive_outcome = (f"{naive_supplier.name}: {stock_before} → {stock_after} units "
                         f"(own threshold = {threshold}) "
                         f"{'⚠ NEW shortage created' if stock_after < threshold else ''}")
    else:
        naive_outcome = "No supplier found"

    safe_outcome = f"{safe_supplier.name} (feasible)" if safe_supplier else "FAILED – no feasible supplier"

    comparison_rows.append({
        'Requesting Hospital': hosp_name,
        'Insulin Needed': qty,
        'OLD naive rule (stock ≥ qty)': naive_outcome,
        'NEW safety-margin rule': safe_outcome,
    })

df_comparison = pd.DataFrame(comparison_rows)
print("Naive vs. Safety-Margin-Aware Supplier Selection — Insulin Requests")
display(df_comparison.style.set_properties(**{'text-align': 'left'}))

print("\nInterpretation: the naive rule would have drained an already below-threshold")
print("supplier even further, silently converting one hospital's shortage into a worse")
print("one elsewhere. The safety-margin rule correctly reports 'no feasible supplier',")
print("which is the honest signal that Insulin needs external procurement / emergency")
print("resupply — an outcome no purely local redistribution algorithm can solve alone")
print("(see Part 1, Sec. 6.6 — the system is a decision-support layer, not a cure-all).")


Naive vs. Safety-Margin-Aware Supplier Selection — Insulin Requests


,Requesting Hospital,Insulin Needed,OLD naive rule (stock ≥ qty),NEW safety-margin rule
0,Matale,20,Kandy: 80 → 60 units (own threshold = 100) ⚠ NEW shortage created,FAILED – no feasible supplier
1,Monaragala,15,Badulla: 25 → 10 units (own threshold = 60) ⚠ NEW shortage created,FAILED – no feasible supplier



Interpretation: the naive rule would have drained an already below-threshold
supplier even further, silently converting one hospital's shortage into a worse
one elsewhere. The safety-margin rule correctly reports 'no feasible supplier',
which is the honest signal that Insulin needs external procurement / emergency
resupply — an outcome no purely local redistribution algorithm can solve alone
(see Part 1, Sec. 6.6 — the system is a decision-support layer, not a cure-all).


## Section 8 – Computational Complexity Analysis

In [16]:
# ─────────────────────────────────────────────────────────────────────────────
# Complexity Analysis – all operations in the system
# ─────────────────────────────────────────────────────────────────────────────

complexity_data = [
    # Operation, Data Structure, Time Complexity, Space Complexity, Notes
    ("add_medicine()",           "HashMap (dict)",      "O(1) avg",          "O(1)",      "Hash table insertion"),
    ("get_stock()",              "HashMap (dict)",      "O(1) avg",          "O(1)",      "Hash table lookup"),
    ("update_stock()",           "HashMap (dict)",      "O(1) avg",          "O(1)",      "Hash table update + deque append"),
    ("urgency_score()",          "HashMap (dict)",      "O(1)",              "O(1)",      "Arithmetic on stored values"),
    ("deque.append()",           "Deque",               "O(1)",              "O(1)",      "Amortised circular buffer"),
    ("deque.popleft()",          "Deque",               "O(1)",              "O(1)",      "FIFO front removal"),
    ("add_route()",              "Adjacency Dict",      "O(1) avg",          "O(1)",      "Two HashMap inserts"),
    ("heapq.heappush()",         "Binary Min-Heap",     "O(log N)",          "O(1)",      "Bubble-up to maintain heap property"),
    ("heapq.heappop()",          "Binary Min-Heap",     "O(log N)",          "O(1)",      "Sink-down after removal"),
    ("build_alert_heap()",       "Binary Min-Heap",     "O(H·M·log(H·M))",   "O(H·M)",   "H=hospitals, M=medicines"),
    ("dijkstra()",               "Min-Heap + HashMap",  "O((V+E) log V)",    "O(V+E)",   "Standard Dijkstra's"),
    ("reconstruct_path()",       "HashMap (pred map)",  "O(V)",              "O(V)",      "Walk predecessor map"),
    ("find_supplier()",          "Min-Heap + HashMap",  "O((V+E) log V)",    "O(V+E)",   "Dominated by Dijkstra"),
    ("transfer()",               "HashMap (dict)",      "O(1)",              "O(1)",      "Two stock updates"),
    ("load_inventory_csv()",     "HashMap",             "O(H·M)",            "O(H·M)",   "H=hospitals, M=medicines"),
    ("load_routes_csv()",        "Adjacency Dict",      "O(E)",              "O(V+E)",   "E=number of route edges"),
    ("load_requests_csv()",      "Deque",               "O(R)",              "O(R)",     "R=number of requests"),
    ("build priority heap",      "Deque → Min-Heap",    "O(R log R)",        "O(R)",     "Drain FIFO arrivals into urgency-ordered heap"),
    ("process_requests()",       "Min-Heap + HashMap",  "O(R log R + R(V+E)log V)", "O(R+V+E)", "R requests popped by priority, each runs Dijkstra"),
]

df_complexity = pd.DataFrame(complexity_data, columns=[
    'Operation', 'Data Structure', 'Time Complexity', 'Space Complexity', 'Notes'
])

print("Computational Complexity of All System Operations:")
display(df_complexity.style
        .set_properties(**{'text-align': 'left'})
        .set_caption("V = vertices (hospitals), E = edges (routes), H = hospitals, M = medicines, R = requests")
        .map(lambda v: 'background-color: #2d5016; color: white'
                  if 'O(1)' in str(v) else
                  ('background-color: #5c2d00; color: white'
                   if 'log' in str(v) else ''),
                  subset=['Time Complexity']))

print("\nKey Complexity Variables:")
print(f"  V (hospitals/nodes)  : {len(system.graph)}")
print(f"  E (routes/edges)     : {sum(len(v) for v in system.graph.values()) // 2}")
print(f"  H (hospital objects) : {len(system.hospitals)}")
print(f"  M (medicine types)   : 5")
print(f"  R (requests)         : 5")
print(f"\nDijkstra theoretical bound: O((V+E) log V) = O(({len(system.graph)}+{sum(len(v) for v in system.graph.values())//2}) × log {len(system.graph)})")


Computational Complexity of All System Operations:


,Operation,Data Structure,Time Complexity,Space Complexity,Notes
0,add_medicine(),HashMap (dict),O(1) avg,O(1),Hash table insertion
1,get_stock(),HashMap (dict),O(1) avg,O(1),Hash table lookup
2,update_stock(),HashMap (dict),O(1) avg,O(1),Hash table update + deque append
3,urgency_score(),HashMap (dict),O(1),O(1),Arithmetic on stored values
4,deque.append(),Deque,O(1),O(1),Amortised circular buffer
5,deque.popleft(),Deque,O(1),O(1),FIFO front removal
6,add_route(),Adjacency Dict,O(1) avg,O(1),Two HashMap inserts
7,heapq.heappush(),Binary Min-Heap,O(log N),O(1),Bubble-up to maintain heap property
8,heapq.heappop(),Binary Min-Heap,O(log N),O(1),Sink-down after removal
9,build_alert_heap(),Binary Min-Heap,O(H·M·log(H·M)),O(H·M),"H=hospitals, M=medicines"



Key Complexity Variables:
  V (hospitals/nodes)  : 11
  E (routes/edges)     : 12
  H (hospital objects) : 10
  M (medicine types)   : 5
  R (requests)         : 5

Dijkstra theoretical bound: O((V+E) log V) = O((11+12) × log 11)


### 8.1 Complexity Growth Visualisation

In [17]:
# ─────────────────────────────────────────────────────────────────────────────
# Visualise how algorithm complexity scales with network size (V = nodes).
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for headless execution

n_values = np.arange(1, 200)

fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor('#1a1a2e')
ax.set_facecolor('#16213e')

# Plot growth curves
ax.plot(n_values, np.ones_like(n_values),        label='O(1) – HashMap ops',          color='#27ae60', linewidth=2)
ax.plot(n_values, np.log2(n_values),             label='O(log V) – Heap push/pop',     color='#3498db', linewidth=2)
ax.plot(n_values, n_values,                       label='O(V) – Path reconstruction',   color='#f39c12', linewidth=2)
ax.plot(n_values, n_values * np.log2(n_values),  label="O(V log V) – Dijkstra's",      color='#e74c3c', linewidth=2, linestyle='--')
ax.plot(n_values, n_values ** 2,                  label='O(V²) – Brute force (naive)',  color='#8e44ad', linewidth=1.5, linestyle=':')

# Mark current system size
v_current = len(system.graph)
ax.axvline(x=v_current, color='white', linestyle='--', alpha=0.5, linewidth=1)
ax.text(v_current + 2, 600, f'Current\nV={v_current}', color='white', fontsize=9)

ax.set_xlim(1, 150)
ax.set_ylim(0, 800)
ax.set_xlabel('Number of Hospitals (V)', color='white', fontsize=11)
ax.set_ylabel('Relative Operation Count', color='white', fontsize=11)
ax.set_title("Algorithm Complexity Growth – Sri Lanka Hospital Network",
             color='white', fontsize=12, pad=10)
ax.legend(facecolor='#2d2d5e', labelcolor='white', fontsize=9, loc='upper left')
ax.tick_params(colors='white')
for spine in ax.spines.values():
    spine.set_color('#4a4a8a')

plt.savefig("complexity_chart.png", dpi=120, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()
plt.close()
print("✅ Complexity chart saved as complexity_chart.png")


✅ Complexity chart saved as complexity_chart.png


/var/folders/c4/p5_t0y6x7jg16h0gg8x7kwx80000gn/T/ipykernel_12945/2572201424.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 8.2 Empirical Runtime Validation

The table and growth chart above state the *theoretical* Big-O bounds.
To satisfy the brief's requirement to "explain how you arrived at your
conclusions", this section **measures actual wall-clock runtime** on
synthetically generated hospital networks of increasing size and checks
that it grows the way the theory predicts — rather than asserting the
complexity on paper alone.

In [18]:
# ─────────────────────────────────────────────────────────────────────────────
# Empirically benchmark Dijkstra's algorithm, Binary Min-Heap operations
# and HashMap operations against increasing input size, to validate the
# theoretical complexity bounds claimed in Section 8.
# ─────────────────────────────────────────────────────────────────────────────
import time
import random
import numpy as np  # already imported in 8.1, re-imported here for a self-contained cell


def build_random_graph(n_nodes: int, edge_prob: float = 0.05, seed: int = 42) -> dict:
    """
    Generate a connected weighted graph with n_nodes for benchmarking.
    A backbone path guarantees connectivity; extra random edges add the
    branching structure a real hospital network would have.
    Time: O(n_nodes^2) to consider all possible edges (benchmark-only
    helper, not part of the production system).
    """
    rng = random.Random(seed)
    nodes = [f"H{i}" for i in range(n_nodes)]
    graph = {node: {} for node in nodes}
    for i in range(n_nodes - 1):                      # connectivity backbone
        w = rng.randint(5, 200)
        graph[nodes[i]][nodes[i + 1]] = w
        graph[nodes[i + 1]][nodes[i]] = w
    for i in range(n_nodes):                          # extra random edges
        for j in range(i + 2, n_nodes):
            if rng.random() < edge_prob:
                w = rng.randint(5, 200)
                graph[nodes[i]][nodes[j]] = w
                graph[nodes[j]][nodes[i]] = w
    return graph


sizes = [10, 25, 50, 100, 200, 400, 800]
dijkstra_times, edge_counts = [], []

for n in sizes:
    g = build_random_graph(n)
    e = sum(len(v) for v in g.values()) // 2
    edge_counts.append(e)
    reps = 5
    t0 = time.perf_counter()
    for _ in range(reps):
        dijkstra(g, "H0")
    t1 = time.perf_counter()
    dijkstra_times.append((t1 - t0) / reps)

df_bench = pd.DataFrame({
    'V (nodes)': sizes, 'E (edges)': edge_counts,
    'Measured Dijkstra Time (s)': dijkstra_times,
})
print("Measured Dijkstra runtime vs. network size:")
display(df_bench.style.format({'Measured Dijkstra Time (s)': '{:.6f}'}))

# Compare the SHAPE of the measured curve against the theoretical
# (V+E) log V curve — correlation close to 1.0 supports the O((V+E) log V)
# claim; it does not (and cannot) prove it, since a handful of sample
# points can be consistent with more than one growth curve, but a low
# correlation would be strong evidence AGAINST the claimed bound.
theoretical = [ (v + e) * np.log2(v) for v, e in zip(sizes, edge_counts) ]
correlation = np.corrcoef(dijkstra_times, theoretical)[0, 1]
print(f"\nCorrelation between measured time and theoretical O((V+E) log V): "
      f"r = {correlation:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#1a1a2e')
for ax in axes:
    ax.set_facecolor('#16213e')
    ax.tick_params(colors='white')
    for spine in ax.spines.values():
        spine.set_color('#4a4a8a')

axes[0].plot(sizes, dijkstra_times, 'o-', color='#e74c3c', label='Measured time')
axes[0].set_xlabel('V (nodes)', color='white')
axes[0].set_ylabel('Time (seconds)', color='white')
axes[0].set_title("Measured Dijkstra Runtime", color='white')
axes[0].legend(facecolor='#2d2d5e', labelcolor='white')

norm_measured    = np.array(dijkstra_times) / max(dijkstra_times)
norm_theoretical = np.array(theoretical) / max(theoretical)
axes[1].plot(sizes, norm_measured, 'o-', color='#e74c3c', label='Measured (normalised)')
axes[1].plot(sizes, norm_theoretical, '--', color='#3498db', label='O((V+E) log V) (normalised)')
axes[1].set_xlabel('V (nodes)', color='white')
axes[1].set_ylabel('Normalised magnitude', color='white')
axes[1].set_title(f"Measured vs. Theoretical Shape (r = {correlation:.3f})", color='white')
axes[1].legend(facecolor='#2d2d5e', labelcolor='white')

plt.tight_layout()
plt.savefig("complexity_benchmark.png", dpi=120, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()
print("✅ Benchmark chart saved as complexity_benchmark.png")

# ── O(1) HashMap vs O(log N) Heap: same empirical-validation approach ──────
hashmap_times, heap_times = [], []
n_ops_list = [1000, 5000, 10000, 50000, 100000, 200000]

for n_ops in n_ops_list:
    d = {}
    t0 = time.perf_counter()
    for i in range(n_ops):
        d[i] = i          # HashMap insert – expected O(1) average
        _ = d.get(i, 0)   # HashMap lookup – expected O(1) average
    t1 = time.perf_counter()
    hashmap_times.append((t1 - t0) / n_ops)

    h = []
    t0 = time.perf_counter()
    for i in range(n_ops):
        heapq.heappush(h, i)   # Binary Min-Heap push – expected O(log N)
    t1 = time.perf_counter()
    heap_times.append((t1 - t0) / n_ops)

df_ops = pd.DataFrame({
    'N Operations': n_ops_list,
    'Avg HashMap Op Time (s)': hashmap_times,
    'Avg Heap Push Time (s)': heap_times,
})
print("\nPer-operation cost as N grows (HashMap should stay ~flat; Heap should grow slowly):")
display(df_ops.style.format({'Avg HashMap Op Time (s)': '{:.9f}',
                              'Avg Heap Push Time (s)': '{:.9f}'}))


Measured Dijkstra runtime vs. network size:

,V (nodes),E (edges),Measured Dijkstra Time (s)
0,10,13,0.000011
1,25,31,0.000024
2,50,95,0.000067
3,100,328,0.000167
4,200,1187,0.000468
5,400,4374,0.001361
6,800,16631,0.004208



Correlation between measured time and theoretical O((V+E) log V): r = 0.9969


✅ Benchmark chart saved as complexity_benchmark.png

Per-operation cost as N grows (HashMap should stay ~flat; Heap should grow slowly):


/var/folders/c4/p5_t0y6x7jg16h0gg8x7kwx80000gn/T/ipykernel_12945/4026059531.py:91: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,N Operations,Avg HashMap Op Time (s),Avg Heap Push Time (s)
0,1000,0.000000183,0.000000131
1,5000,0.000000167,0.000000122
2,10000,0.000000162,0.000000114
3,50000,0.000000193,0.000000117
4,100000,0.000000178,0.000000118
5,200000,0.000000173,0.000000113


### 8.3 Shortest-Path Distance Matrix

In [19]:
# ─────────────────────────────────────────────────────────────────────────────
# Compute all-pairs shortest distances by running Dijkstra from every node.
# This creates an N × N distance matrix.
# Time complexity: O(V × (V+E) log V)
# ─────────────────────────────────────────────────────────────────────────────

hospital_nodes = sorted(system.hospitals.keys())

# Run Dijkstra from every hospital node
distance_matrix = {}
for src in hospital_nodes:
    distances, _ = dijkstra(system.graph, src)
    distance_matrix[src] = {dst: distances.get(dst, float('inf'))
                             for dst in hospital_nodes}

# Build DataFrame
df_distances = pd.DataFrame(distance_matrix, index=hospital_nodes,
                             columns=hospital_nodes)
df_distances = df_distances.replace(float('inf'), 9999)

print("All-Pairs Shortest Path Distance Matrix (km):")
display(df_distances.style
        .background_gradient(cmap='RdYlGn_r', vmin=0, vmax=400)
        .format("{:.0f}")
        .set_caption("Distance in km between all hospital pairs (∞ → 9999 = unreachable)"))


All-Pairs Shortest Path Distance Matrix (km):


,Anuradhapura,Badulla,Hambantota,Kandy,Kurunegala,Matale,Monaragala,Nuwara Eliya,Polonnaruwa,Ratnapura
Anuradhapura,0,220,360,130,88,110,278,205,105,222
Badulla,220,0,140,350,308,330,58,425,115,280
Hambantota,360,140,0,232,274,257,82,307,255,140
Kandy,130,350,232,0,42,25,314,75,235,92
Kurunegala,88,308,274,42,0,67,356,117,193,134
Matale,110,330,257,25,67,0,339,100,215,117
Monaragala,278,58,82,314,356,339,0,389,173,222
Nuwara Eliya,205,425,307,75,117,100,389,0,310,167
Polonnaruwa,105,115,255,235,193,215,173,310,0,327
Ratnapura,222,280,140,92,134,117,222,167,327,0


## Section 9 – Responsible Use of Generative AI

### 9.1 GenAI Tools Used
- **Tool:** Claude (Anthropic) / ChatGPT (OpenAI)
- **Purpose:** Idea generation, algorithm explanation review, documentation drafting

### 9.2 Prompts Used

1. *"Explain how Dijkstra's algorithm works for finding shortest paths in a hospital network. What data structures does it use internally?"*
2. *"What is the time and space complexity of Dijkstra's SSSP algorithm using a binary min-heap?"*
3. *"How can a min-heap (heapq in Python) be used to implement a priority queue for low-stock alerts in a medical supply chain system?"*
4. *"What is the difference between a deque and a regular list for implementing a FIFO request queue? What are the time complexity advantages?"*
5. *"Part 1 argues that first-come-first-served allocation is inappropriate for medical urgency, and that a surplus facility must keep its own safety stock before donating. Does the current process_requests()/find_supplier() implementation actually enforce either of those rules, or does it only claim to in the report?"*

### 9.3 Process by Which GenAI Supported This Work

GenAI was used as an **assistive tool** in the following ways:

| Stage | GenAI Role | Human Contribution |
|---|---|---|
| Algorithm selection | Confirmed Dijkstra's suitability | Analysed problem constraints |
| Complexity analysis | Provided initial Big-O descriptions | Verified against textbook sources |
| Code documentation | Suggested docstring phrasing | Reviewed, edited, and validated all content |
| Visualisation | Suggested chart types | Designed and implemented custom plots |
| Traceability check | Flagged that the original `process_requests()` used plain FIFO and `find_supplier()` had no safety-stock check, contradicting Part 1 Secs. 3.3/5.5 | Verified the gap against the Part 1 text, designed and implemented the Min-Heap priority scheduler and the safety-margin feasibility test, and re-ran the dataset to confirm the fix changes real output (Section 7.2) |

> **Important:** All code was written, tested, and debugged manually. GenAI did not write code autonomously. Every suggestion was critically reviewed against course materials and academic references.

### 9.4 Academic References

1. Ministry of Health, Sri Lanka. (2024). *Analysis of Sri Lanka's Policy on Healthcare Delivery for Universal Health Coverage 2025*. Colombo: Ministry of Health.
2. Cormen, T. H., Leiserson, C. E., Rivest, R. L., & Stein, C. (2022). *Introduction to Algorithms* (4th ed.). MIT Press. [Chapter 24 – Single-Source Shortest Paths]
3. IJSRA. (2025). *Optimizing supply chain efficiency in healthcare using machine learning*. International Journal of Scientific Research in Agricultural Sciences.


## Section 10 – Conclusion

This notebook implements the computational solution proposed in Part 1's
"Ceylon Medical Grid" report and answers its research question — *how can
data structures and graph-based algorithms be used to prioritise medicine
shortages and efficiently redistribute available supplies across Sri
Lanka's healthcare network* — with a working, individually-authored
Python system rather than a purely theoretical design.

**What was implemented.** A weighted graph (adjacency-dict) models the
hospital network; a HashMap (`dict`) gives O(1) average inventory
lookup/update per hospital; a Binary Min-Heap drives two independent
priority mechanisms — low-stock alerting (Section 5) and, critically,
**request processing order** (Section 6); and Dijkstra's algorithm
finds the minimum-cost route between a critical hospital and a feasible
supplier (Section 2.2). A `deque` still serves as the FIFO arrival
buffer for incoming requests, but — unlike a naive first-come,
first-served system — it is drained into the priority Min-Heap before
being served, directly addressing the FCFS critique raised in Part 1,
Section 3.3.

**What the implementation revealed.** Two refinements were made beyond
Part 1's original description, and both changed the system's actual
output, not just its internal bookkeeping:

1. **Priority-ordered processing** (Section 6) changed the *order* in
   which requests are served — the two Insulin requests, which had the
   lowest urgency scores in the dataset, are now handled first instead
   of whatever position they happened to occupy in the CSV.
2. **Safety-margin-aware supplier selection** (Section 7.2) changed
   *whether* a request could succeed at all. Because every hospital's
   own Insulin stock is already below its own safety-stock threshold in
   this dataset, the system correctly reports both Insulin requests as
   `FAILED – no supplier`, rather than draining an already-critical
   hospital to mask the shortage elsewhere. The naive version of the
   algorithm (Section 7.2's comparison) would have done exactly that.

This second result is, in a sense, the most important finding in the
notebook: it demonstrates that a correctly-designed redistribution
algorithm does not just move medicine faster — it can also correctly
detect when local redistribution *cannot* solve a shortage, which is
precisely the boundary condition Part 1, Section 6.6 argued the system
must respect (a decision-support layer, not an autonomous authority
that hides problems by producing an optimistic-looking transfer).

**Complexity.** Section 8 both derives and empirically validates the
system's complexity bounds. All inventory operations are O(1) average
(HashMap); heap push/pop is O(log N); and the dominant cost,
`process_requests()`, is O(R log R + R·(V+E) log V) — R priority-heap
operations plus R Dijkstra searches. The empirical benchmark (Section
8.2) measured this scaling on synthetic networks up to 800 nodes and
found the measured runtime shape closely tracks the theoretical
O((V+E) log V) curve, giving evidence for the complexity claim beyond
the static Big-O table alone.

**Limitations and future work.** The current model uses a single
minimum-stock threshold per hospital across all medicines, rather than
a per-medicine threshold — a simplification of Part 1's proposed
composite priority function (which also envisaged population, clinical
urgency and accessibility weighting). `find_supplier()` also selects a
single best supplier per request rather than splitting a request across
multiple partial suppliers, and does not yet re-run the low-stock alert
heap after each transfer to reflect newly-created shortages at the
supplying hospital. These are natural extensions rather than flaws in
the present scope, and none of them change the complexity bounds
established above, since each would still be bounded by a small
constant factor times the existing Dijkstra/heap operations.

Overall, the notebook shows that the data structures and algorithms
proposed in Part 1 — HashMap, Min-Heap, weighted graph and Dijkstra's
algorithm — are not only theoretically appropriate for Sri Lanka's
medicine-redistribution problem, but produce concrete, explainable and
occasionally uncomfortable results (an unfulfillable request) that a
purely descriptive proposal could not have surfaced.

In [20]:
# ─────────────────────────────────────────────────────────────────────────────
# Final system summary and validation
# ─────────────────────────────────────────────────────────────────────────────

print("=" * 65)
print("  COM713 Part 2 – System Summary")
print("=" * 65)
print(f"\n  Network Statistics:")
print(f"    Hospitals in network      : {len(system.hospitals)}")
print(f"    Transport routes          : {sum(len(v) for v in system.graph.values())//2}")
print(f"    Medicine types tracked    : 5")
print(f"    Requests processed        : {len(results)}")
print(f"    Successful transfers      : {sum(1 for r in results if r['status']=='SUCCESS')}")
print(f"\n  Data Structures Demonstrated:")
print(f"    ✅ HashMap (dict)         – inventory, distances, predecessors")
print(f"    ✅ Weighted Graph         – hospital route network")
print(f"    ✅ Binary Min-Heap        – low-stock alerts AND request-priority queue")
print(f"    ✅ Deque (FIFO)           – request arrival buffer (drained into the heap)")
print(f"    ✅ List                   – transfer audit trail")
print(f"\n  Algorithms Implemented:")
print(f"    ✅ Dijkstra's SSSP        – O((V+E) log V)")
print(f"    ✅ Path Reconstruction    – O(V)")
print(f"    ✅ Safety-Margin Supplier Select – O(H) = O(V), rejects unsafe transfers")
print(f"    ✅ Priority-Queue Request Scheduling – O(R log R + R·(V+E) log V)")
print(f"    ✅ Min-Heap Alert Build   – O(H·M log H·M)")
print(f"\n  OOP Design:")
print(f"    ✅ Hospital class         – encapsulates inventory + history")
print(f"    ✅ SupplyChain class      – orchestrates graph + algorithms")
print("=" * 65)
print("  ✅ All systems operational. Assignment Part 2 complete.")
print("=" * 65)


  COM713 Part 2 – System Summary

  Network Statistics:
    Hospitals in network      : 10
    Transport routes          : 12
    Medicine types tracked    : 5
    Requests processed        : 5
    Successful transfers      : 3

  Data Structures Demonstrated:
    ✅ HashMap (dict)         – inventory, distances, predecessors
    ✅ Weighted Graph         – hospital route network
    ✅ Binary Min-Heap        – low-stock alerts AND request-priority queue
    ✅ Deque (FIFO)           – request arrival buffer (drained into the heap)
    ✅ List                   – transfer audit trail

  Algorithms Implemented:
    ✅ Dijkstra's SSSP        – O((V+E) log V)
    ✅ Path Reconstruction    – O(V)
    ✅ Safety-Margin Supplier Select – O(H) = O(V), rejects unsafe transfers
    ✅ Priority-Queue Request Scheduling – O(R log R + R·(V+E) log V)
    ✅ Min-Heap Alert Build   – O(H·M log H·M)

  OOP Design:
    ✅ Hospital class         – encapsulates inventory + history
    ✅ SupplyChain class      – orch